# Python Async programming


In [23]:
import asyncio
import time

In [24]:
# Load asynchronous processing functions

async def load_proc(id: int, wait_time: int, fail: bool = False):
    # Your asynchronous processing code here
    try:
        print(f"Starting async process {id} with wait time: {time.monotonic()}")
        await asyncio.sleep(wait_time)
        print(f"Finished async process {id} with wait time: {time.monotonic()}")    
    except asyncio.CancelledError as e:
        print(f"Process {id} was cancelled: {e}")
        return f"Process {id} failed"
    if fail:
        raise Exception("Forced failure")
    return f"Process {id} finished"

In [25]:
start_time = time.monotonic()
proc_id01, proc_id02 = await asyncio.gather(load_proc(1, 2), load_proc(2, 3))
stop_time = time.monotonic()

print(f"Total time: {stop_time - start_time}")
print(f"Proc. IDs: {proc_id01}, {proc_id02}")


Starting async process 1 with wait time: 431654.516427916
Starting async process 2 with wait time: 431654.516467666
Finished async process 1 with wait time: 431656.517632291
Finished async process 2 with wait time: 431657.517716541
Total time: 3.002479917020537
Proc. IDs: Process 1 finished, Process 2 finished


In [26]:
# Multiple calls
start_time = time.monotonic()
ids = await asyncio.gather(*(load_proc(id, 2) for id in range(1, 4)))
stop_time = time.monotonic()

print(f"Total time: {stop_time - start_time}")
print(f"Proc. IDs: {ids}")

Starting async process 1 with wait time: 431657.531180708
Starting async process 2 with wait time: 431657.531269333
Starting async process 3 with wait time: 431657.531284
Finished async process 1 with wait time: 431659.53228325
Finished async process 2 with wait time: 431659.5327435
Finished async process 3 with wait time: 431659.532774666
Total time: 2.0033454580116086
Proc. IDs: ['Process 1 finished', 'Process 2 finished', 'Process 3 finished']


In [27]:
# Concurrents tasks example

# Create a task
task = asyncio.create_task(load_proc(1, 1, fail=False))
print(f"[BEFORE await] Task is Done? {task.done()}")
result = await task
print(f"[AFTER await] Task is Done? {task.done()}")
print(f"Result: {result}")


[BEFORE await] Task is Done? False
Starting async process 1 with wait time: 431659.554341541
Finished async process 1 with wait time: 431660.555566708
[AFTER await] Task is Done? True
Result: Process 1 finished


In [28]:
# Fail programatically
task = asyncio.create_task(load_proc(2, 1, fail=True))
print(f"[BEFORE await] Task is Done? {task.done()}")
try:
    result = await task
    print(f"Result: {result}")
except Exception as e:
    print(f"Task failed with exception: {e}")
print(f"[AFTER await] Task is Done? {task.done()}")

[BEFORE await] Task is Done? False
Starting async process 2 with wait time: 431660.56878
Finished async process 2 with wait time: 431661.569952125
Task failed with exception: Forced failure
[AFTER await] Task is Done? True


In [29]:
# Cancel a task example
tasks =[asyncio.create_task(load_proc(id, 5, fail=False)) for id in range(3)]
statuses = [task.done() for task in tasks]
print(f"[BEFORE cancel] Task is Done?")
print(f"Statuses: {statuses}")
try:
    await asyncio.sleep(1)  # Simulate some work before cancelling the task
    for task in tasks:
        task.cancel(msg="Cancelling task after 1 second.")
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    for result, task in zip(results, tasks):
        print(f"Result: {result}, Task Done: {task.get_name()}")
    # print(f"Results: {results}")
except asyncio.CancelledError:
    print("Task was cancelled")
print(f"[AFTER cancel] Task is Done?")
statuses = [task.done() for task in tasks]
print(f"Statuses: {statuses}")


[BEFORE cancel] Task is Done?
Statuses: [False, False, False]
Starting async process 0 with wait time: 431661.577810541
Starting async process 1 with wait time: 431661.577847
Starting async process 2 with wait time: 431661.577856875
Process 0 was cancelled: Cancelling task after 1 second.
Process 1 was cancelled: Cancelling task after 1 second.
Process 2 was cancelled: Cancelling task after 1 second.
Result: Process 0 failed, Task Done: Task-278
Result: Process 1 failed, Task Done: Task-279
Result: Process 2 failed, Task Done: Task-280
[AFTER cancel] Task is Done?
Statuses: [True, True, True]


In [30]:
# Implement timetouts pattern

tasks = [asyncio.wait_for(asyncio.sleep(wait_time), timeout=2) for wait_time in range(5)]

for task in tasks:
    try:
        print("Starting timeout test")
        await task
        print("Timeout test completed successfully")
    except asyncio.TimeoutError:
        print("Operation timed out")
    
# Call load_proc with timeout
async def load_proc_with_timeout(id, wait_time):
    print(f"Starting load_proc with wait_time={wait_time}")
    try:
        async with asyncio.timeout(2):
            await load_proc(id, wait_time)
        print("Load proc completed successfully")
    except asyncio.TimeoutError:
        print("Load proc timed out")
    
# Example usage of load_proc_with_timeout
print("=" * 10)
for id, wait_time in zip(range(5), range(5)):
    await load_proc_with_timeout(id, wait_time)


Starting timeout test
Timeout test completed successfully
Starting timeout test
Timeout test completed successfully
Starting timeout test
Operation timed out
Starting timeout test
Operation timed out
Starting timeout test
Operation timed out
Starting load_proc with wait_time=0
Starting async process 0 with wait time: 431669.602983708
Finished async process 0 with wait time: 431669.603410541
Load proc completed successfully
Starting load_proc with wait_time=1
Starting async process 1 with wait time: 431669.603456041
Finished async process 1 with wait time: 431670.604812791
Load proc completed successfully
Starting load_proc with wait_time=2
Starting async process 2 with wait time: 431670.605081416
Process 2 was cancelled: 
Load proc completed successfully
Starting load_proc with wait_time=3
Starting async process 3 with wait time: 431672.606772291
Process 3 was cancelled: 
Load proc completed successfully
Starting load_proc with wait_time=4
Starting async process 4 with wait time: 43167

In [31]:
# Queue example

tasks_queue = asyncio.Queue()

producers = [asyncio.create_task(tasks_queue.put(f"task1-{i}")) for i in range(5)]

async def consumers():
    while True:
        task = await tasks_queue.get()
        if str(task).endswith("-4") is True:
            tasks_queue.task_done()
            break
        print(f"Consumed {task}")
        tasks_queue.task_done()
        
consumer = asyncio.create_task(consumers())

await asyncio.gather(*producers, consumer)

print("All tasks have been processed.")


Consumed task1-0
Consumed task1-1
Consumed task1-2
Consumed task1-3
All tasks have been processed.


In [32]:
# Backpressure 

tasks_queue = asyncio.Queue(maxsize=2)

async def producer(name):
    for i in range(5):
        await tasks_queue.put(f"task-{name}-{i}")
        print(f"Produced task-{name}-{i}")
        await asyncio.sleep(.1)  # Simulate some delay for backpressure effect

async def consumer(name):
    while True:
        task = await tasks_queue.get()
        if str(task).endswith("-4") is True:
            tasks_queue.task_done()
            break
        print(f"Consumer {name} consumed {task}")
        tasks_queue.task_done()
        await asyncio.sleep(2)  # Simulate some processing delay
        
await asyncio.gather(
    producer("A"),
    producer("B"),
    consumer("X")
)
print("All tasks have been processed.")

        

Produced task-A-0
Produced task-B-0
Consumer X consumed task-A-0
Produced task-A-1
Consumer X consumed task-B-0
Produced task-B-1
Consumer X consumed task-A-1
Produced task-A-2
Consumer X consumed task-B-1
Produced task-B-2
Consumer X consumed task-A-2
Produced task-A-3
Consumer X consumed task-B-2
Produced task-B-3
Consumer X consumed task-A-3
Produced task-A-4
Consumer X consumed task-B-3
Produced task-B-4
All tasks have been processed.


In [33]:
# Priority Queue

tasks_queue = asyncio.PriorityQueue()

await tasks_queue.put((3, "task-3"))
await tasks_queue.put((1, "task-1"))
await tasks_queue.put((2, "task-2"))
await tasks_queue.put((0, "task-0"))

while not tasks_queue.empty():
    priority, task = await tasks_queue.get()
    print(f"Consumed {task} with priority {priority}")
    tasks_queue.task_done()
    
await tasks_queue.join()
print("All tasks have been processed.")

Consumed task-0 with priority 0
Consumed task-1 with priority 1
Consumed task-2 with priority 2
Consumed task-3 with priority 3
All tasks have been processed.


In [34]:
# Sync. primitives

counter = 0

async def counter_unsave():
    global counter
    local_counter = counter
    local_counter += 1
    await asyncio.sleep(0.1)
    counter = local_counter
    
await asyncio.gather(*(counter_unsave() for _ in range(10)))

print("Unsaved Counter = ", counter)

# Save primitive
lock = asyncio.Lock()
counter = 0

async def counter_save():
    global counter
    async with lock:
        local_counter = counter
        local_counter += 1
        await asyncio.sleep(0.1)
        counter = local_counter
    
await asyncio.gather(*(counter_save() for _ in range(10)))

print("Saved Counter = ", counter)

Unsaved Counter =  1
Saved Counter =  10


In [35]:
# Events

event = asyncio.Event()

async def waiter():
    print(f"Waiting for event... {event.is_set()}")
    await event.wait()
    print("Event received! ", f"{event.is_set()}")

async def setter():
    print(f"Setting event... {event.is_set()}")
    await asyncio.sleep(1)
    event.set()
    print("Event set! ", f"{event.is_set()}")

await asyncio.gather(*(asyncio.gather(waiter(), setter()) for _ in range(5)))

print("Event handling complete.")

Waiting for event... False
Setting event... False
Waiting for event... False
Setting event... False
Waiting for event... False
Setting event... False
Waiting for event... False
Setting event... False
Waiting for event... False
Setting event... False
Event set!  True
Event set!  True
Event set!  True
Event set!  True
Event set!  True
Event received!  True
Event received!  True
Event received!  True
Event received!  True
Event received!  True
Event handling complete.


In [36]:
# Conditions

condition = asyncio.Condition()

order = {"step": 0}

async def step(n):
    print(f"Waiting for step {n}...")
    print(f"Current order step: {order['step']}")
    async with condition:
        await condition.wait_for(lambda: order["step"] >= n)
        print(f"Step {n} executed")
        order["step"] -= n
        print(f"Order step after executing step {n}: {order['step']}")        

# Add steps to the order
async def produce_steps():
    for n in range(6):
        async with condition:
            order["step"] += n
            condition.notify_all()
            await asyncio.sleep(.5)
            print(f"Order step after adding step {n}: {order['step']}")

await asyncio.gather(produce_steps(), *(step(n) for n in range(5)))

print("Condition handling complete.")


Waiting for step 0...
Current order step: 0
Waiting for step 1...
Current order step: 0
Waiting for step 2...
Current order step: 0
Waiting for step 3...
Current order step: 0
Waiting for step 4...
Current order step: 0
Order step after adding step 0: 0
Step 0 executed
Order step after executing step 0: 0
Order step after adding step 1: 1
Step 1 executed
Order step after executing step 1: 0
Order step after adding step 2: 2
Step 2 executed
Order step after executing step 2: 0
Order step after adding step 3: 3
Step 3 executed
Order step after executing step 3: 0
Order step after adding step 4: 4
Step 4 executed
Order step after executing step 4: 0
Order step after adding step 5: 5
Condition handling complete.


In [37]:
# Semaphore example

machines = asyncio.Semaphore(3)
products_queue = asyncio.Queue()
lock = asyncio.Lock()

MAX_PRODUCTS = 10
consumed_count = 0

async def produce(item):
    async with machines:
        await products_queue.put(f"Product {item}")
        print(f"Produced Product {item}")
        await asyncio.sleep(0.5)
        
async def consume():
    while True:
        async with lock:
            product = await products_queue.get()
            print(f"Consumed {product}")
            global consumed_count
            products_queue.task_done()
            consumed_count += 1
            if consumed_count >= MAX_PRODUCTS:
                break
            # await asyncio.sleep(0.1)

await asyncio.gather(
    *(produce(i) for i in range(MAX_PRODUCTS)),
    consume()
)

print(f"Consumed {consumed_count} products")


Produced Product 0
Produced Product 1
Produced Product 2
Consumed Product 0
Consumed Product 1
Consumed Product 2
Produced Product 3
Produced Product 4
Produced Product 5
Consumed Product 3
Consumed Product 4
Consumed Product 5
Produced Product 6
Produced Product 7
Produced Product 8
Consumed Product 6
Consumed Product 7
Consumed Product 8
Produced Product 9
Consumed Product 9
Consumed 10 products


In [38]:
# Asyncio Semaphore 2

semaphore = asyncio.Semaphore(2)

async def acquire_semaphore(item):
    print(f"Acquiring semaphore... {item}")
    await semaphore.acquire()
    print(f"Semaphore acquired {item}")
    
async def release_semaphore(item):
    print(f"Releasing semaphore... {item}")
    semaphore.release()
    print(f"Semaphore released {item}")

await asyncio.gather(
    acquire_semaphore(1),
    acquire_semaphore(2),
    acquire_semaphore(3),
    release_semaphore(1),
    release_semaphore(2),
    release_semaphore(3)
)

print("All tasks completed")


Acquiring semaphore... 1
Semaphore acquired 1
Acquiring semaphore... 2
Semaphore acquired 2
Acquiring semaphore... 3
Releasing semaphore... 1
Semaphore released 1
Releasing semaphore... 2
Semaphore released 2
Releasing semaphore... 3
Semaphore released 3
Semaphore acquired 3
All tasks completed


In [43]:
# Semaphore Example 3

semaphore = asyncio.BoundedSemaphore(2)

async def acquire_semaphore(item):
    print(f"Acquiring semaphore... {item}")
    await semaphore.acquire()
    print(f"Semaphore acquired {item}")

async def release_semaphore(item):
    print(f"Releasing semaphore... {item}")
    semaphore.release()
    print(f"Semaphore released {item}")
    
try:
    await asyncio.gather(
        acquire_semaphore(1),
        acquire_semaphore(2),
        acquire_semaphore(3),
        release_semaphore(1),
        release_semaphore(2),
        release_semaphore(3),
        release_semaphore(4),
    )
except ValueError as e:
    print(f"Error: {e}")

print("All tasks completed")

Acquiring semaphore... 1
Semaphore acquired 1
Acquiring semaphore... 2
Semaphore acquired 2
Acquiring semaphore... 3
Releasing semaphore... 1
Semaphore released 1
Releasing semaphore... 2
Semaphore released 2
Releasing semaphore... 3
Semaphore released 3
Releasing semaphore... 4
Semaphore acquired 3
Error: BoundedSemaphore released too many times
All tasks completed


In [ ]:
# Task groups

